In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

# ------------ Paths ------------
file1 = r"D:\ChipVista\Projects\decaying fruits\DATALOG-1.csv"  # 2025-09-30 → 2025-10-03 12:00 (minute-cycle format)
file2 = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"    # 2025-10-04 → 2025-10-08 (full datetime)
save_folder = r"D:\ChipVista\Projects\decaying fruits"
os.makedirs(save_folder, exist_ok=True)

# ------------ Time windows ------------
F1_START = "2025-09-30 00:00:00"
F1_END   = "2025-10-03 12:00:00"
F2_START = "2025-10-04 00:00:00"
F2_END   = "2025-10-08 23:59:59"   # adjust if you want a tighter end
TIME_LABEL = "%Y-%m-%d %H:%M"

# ================== Part A: DATALOG-1.csv (hour detection) ==================
df1 = pd.read_csv(file1)[['mq2','mq4','Time']]

# Detect hour by minute wraparound
hours = []
current_hour = 0
prev_min = None
for t in df1['Time']:
    try:
        mins, _ = t.split(":")
        mins = int(mins)
    except Exception:
        mins = None
    if prev_min is not None and mins is not None and mins < prev_min:
        current_hour += 1
    hours.append(current_hour)
    prev_min = mins

df1['HourIndex'] = hours

# Hourly mean and map to real datetime starting from F1_START
hourly1 = df1.groupby('HourIndex')[['mq2','mq4']].mean().reset_index()
start_dt1 = pd.to_datetime(F1_START)
hourly1['dt'] = start_dt1 + pd.to_timedelta(hourly1['HourIndex'], unit='h')
hourly1 = hourly1[['dt','mq2','mq4']].set_index('dt')
# (Optional) clip to the declared end
hourly1 = hourly1.loc[:pd.to_datetime(F1_END)]

# ================== Part B: DATALOG.csv (direct datetime) ===================
df2 = pd.read_csv(file2)[['mq2','mq4','Time']]
df2['Time'] = pd.to_datetime(df2['Time'], format='%d/%m/%Y %H:%M', errors='coerce')
df2 = df2.dropna(subset=['Time'])
# Filter desired window
df2 = df2[(df2['Time'] >= pd.to_datetime(F2_START)) & (df2['Time'] <= pd.to_datetime(F2_END))]
# Resample to hourly mean to match file1 granularity
hourly2 = (
    df2.set_index('Time')
       .sort_index()
       .resample('1H')
       .mean()
)

# ================== Part C: Combine & Plot ===================
combined = pd.concat([hourly1, hourly2], axis=0).sort_index()

overall_start = pd.to_datetime(F1_START)
overall_end   = pd.to_datetime(F2_END)
tick_positions = pd.date_range(start=overall_start, end=overall_end, freq='12H')

def plot_series(df, col, color, title, filename, y_label):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df.index, df[col], marker='o', linestyle='-', color=color, label=f"{col.upper()} (ppm)")
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel(y_label)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend()

    # Ticks every 12 hours with full date+year
    ax.set_xticks(tick_positions)
    ax.xaxis.set_major_formatter(mdates.DateFormatter(TIME_LABEL))
    plt.xticks(rotation=30, ha='right')

    # Add ~5% horizontal margin so the line doesn't touch the frame
    total_range = overall_end - overall_start
    margin = total_range * 0.05
    ax.set_xlim(overall_start - margin, overall_end + margin)

    plt.tight_layout()
    fig.savefig(os.path.join(save_folder, filename), dpi=300)
    plt.close(fig)

plot_series(
    combined, 'mq2', 'blue',
    "Overall MQ2 Concentration (ppm)\n(2025-09-30 → 2025-10-08, hourly means; ticks every 12h)",
    "Overall_MQ2_FULL.png", "Concentration (ppm)"
)

plot_series(
    combined, 'mq4', 'green',
    "Overall MQ4 Concentration (ppm)\n(2025-09-30 → 2025-10-08, hourly means; ticks every 12h)",
    "Overall_MQ4_FULL.png", "Concentration (ppm)"
)

print("✅ Combined MQ2/MQ4 graphs generated:", 
      os.path.join(save_folder, "Overall_MQ2_FULL.png"),
      os.path.join(save_folder, "Overall_MQ4_FULL.png"))


C:\Users\mbila\AppData\Local\Temp\ipykernel_41592\3838628281.py:57: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample('1H')
C:\Users\mbila\AppData\Local\Temp\ipykernel_41592\3838628281.py:66: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  tick_positions = pd.date_range(start=overall_start, end=overall_end, freq='12H')


✅ Combined MQ2/MQ4 graphs generated: D:\ChipVista\Projects\decaying fruits\Overall_MQ2_FULL.png D:\ChipVista\Projects\decaying fruits\Overall_MQ4_FULL.png


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os

# ==== File Paths ====
file_exp1 = r"D:\ChipVista\Projects\decaying fruits\EXP_01_MEGA_UNO_updated.csv"
file_exp2 = r"D:\ChipVista\Projects\decaying fruits\EXP_02_UNO_updated.csv"
save_folder = os.path.dirname(file_exp1)

# ==== Read CSVs ====
df1 = pd.read_csv(file_exp1)
df2 = pd.read_csv(file_exp2)

print("EXP1 Columns:", df1.columns.tolist())
print("EXP2 Columns:", df2.columns.tolist())

# ==== Generate Time Columns ====
start_datetime_exp1 = datetime(2025, 10, 17, 17, 0, 0)
start_datetime_exp2 = datetime(2025, 10, 17, 17, 0, 0)

df1["Time"] = [start_datetime_exp1 + timedelta(minutes=10 * i) for i in range(len(df1))]
df2["Time"] = [start_datetime_exp2 + timedelta(minutes=10 * i) for i in range(len(df2))]

# ==== Plot 1: MQ2 Comparison ====
plt.figure(figsize=(10, 5))
plt.plot(df1["Time"], df1["mq2_ppm"], marker="o", linewidth=2, label="Experiment 1 - MQ2")
plt.plot(df2["Time"], df2["mq2_ppm"], marker="s", linewidth=2, label="Experiment 2 - MQ2", color="green")
plt.title("MQ2 PPM vs Time (Experiment 1 vs Experiment 2)")
plt.xlabel("Time (starting from 17/10/2025)")
plt.ylabel("MQ2 PPM")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()

mq2_combined_path = os.path.join(save_folder, "MQ2_Combined_EXP1_EXP2.png")
plt.savefig(mq2_combined_path, dpi=300)
print(f"✅ Combined MQ2 graph saved at: {mq2_combined_path}")
plt.close()

# ==== Plot 2: MQ4 Comparison ====
plt.figure(figsize=(10, 5))
plt.plot(df1["Time"], df1["mq4_ppm"], color="orange", marker="o", linewidth=2, label="Experiment 1 - MQ4")
plt.plot(df2["Time"], df2["mq4_ppm"], color="red", marker="s", linewidth=2, label="Experiment 2 - MQ4")
plt.title("MQ4 PPM vs Time (Experiment 1 vs Experiment 2)")
plt.xlabel("Time (starting from 17/10/2025)")
plt.ylabel("MQ4 PPM")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()

mq4_combined_path = os.path.join(save_folder, "MQ4_Combined_EXP1_EXP2.png")
plt.savefig(mq4_combined_path, dpi=300)
print(f"✅ Combined MQ4 graph saved at: {mq4_combined_path}")
plt.close()

print("🎯 Combined comparison graphs generated and saved successfully!")


EXP1 Columns: ['Time', 'mq2_ppm', 'mq4_ppm']
EXP2 Columns: ['Time', 'mq2_ppm', 'mq4_ppm']
✅ Combined MQ2 graph saved at: D:\ChipVista\Projects\decaying fruits\MQ2_Combined_EXP1_EXP2.png
✅ Combined MQ4 graph saved at: D:\ChipVista\Projects\decaying fruits\MQ4_Combined_EXP1_EXP2.png
🎯 Combined comparison graphs generated and saved successfully!
